# 🎬 Video Object Replacement — CogVideoX Edition

### Принципиальное отличие от SDXL-версии

| | SDXL-версия | **CogVideoX-версия** |
|---|---|---|
| Модель | SDXL Inpainting (image) | **CogVideoX-2B (video)** |
| Обработка | Кадр за кадром независимо | **Чанки по 49 кадров сразу** |
| Временная связность | Оптический поток (пост-обработка) | **Встроена в архитектуру модели** |
| Мерцание | Есть (особенно на быстром движении) | **Минимальное** |
| Скорость | Быстрее | Медленнее |

**CogVideoX** — видео-диффузионная модель от THUDM. Она обрабатывает сразу 49 кадров (~6 секунд) как единое целое через пространственно-временное внимание, что даёт естественную временную связность без дополнительных трюков.

### Инструкция:
1. **Runtime → Change runtime type → T4 GPU** (обязательно!)
2. Запускай ячейки по порядку
3. Загрузи видео, настрой параметры, запусти пайплайн

> ⚠️ CogVideoX-2B весит ~6 GB и скачивается с HuggingFace (~5-10 мин при первом запуске)

## 0. Проверка GPU

In [1]:
import subprocess

def run(cmd):
    print(f"▶ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    lines = (r.stdout or "").strip().splitlines()
    if lines: print("\n".join(lines[-3:]))
    if r.returncode != 0 and r.stderr:
        print(r.stderr[-1000:])

# transformers==4.44.2 — последняя версия, совместимая и с CogVideoX, и с groundingdino-py.
# Более новые transformers (>=4.45) ломают groundingdino-py (убрали get_head_mask).
run("pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128")
run("pip install xformers")

▶ pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128
▶ pip install xformers


In [2]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU не найден!\n"
        "Перейди: Runtime → Change runtime type → Hardware accelerator → GPU (T4)"
    )

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"✅ GPU: {gpu_name}")
print(f"   VRAM: {vram_gb:.1f} GB")
print(f"   CUDA: {torch.version.cuda}")

if vram_gb < 13:
    print("\n⚠ VRAM < 13 GB. CogVideoX-2B может не влезть.")
    print("  Попробуй включить CPU offload (параметр ENABLE_CPU_OFFLOAD=True ниже).")

✅ GPU: NVIDIA H100 80GB HBM3
   VRAM: 85.0 GB
   CUDA: 12.8


## 1. Установка зависимостей

In [6]:
# --index-url принудительно использует официальный PyPI (а не внутренний зеркальный).
# transformers==4.44.2 — последняя версия, совместимая и с CogVideoX, и с GroundingDINO.
# Более новые transformers (>=4.45) ломают GroundingDINO (убрали get_head_mask).
PYPI = "--index-url https://pypi.org/simple/"
run(f"pip install -q {PYPI} 'diffusers>=0.31.0' 'transformers==4.44.2' accelerate 'imageio[ffmpeg]' opencv-python-headless psutil supervision")

# GroundingDINO — устанавливаем напрямую с GitHub (пакет groundingdino-py может отсутствовать
# во внутренних зеркалах; git-установка всегда берётся с официального репозитория IDEA Research).
run(f"pip install -q {PYPI} 'groundingdino-py'")

# SAM2 — тоже с GitHub (нет на PyPI)
run("pip install -q 'git+https://github.com/facebookresearch/sam2.git'")

print("\n✅ Зависимости установлены!")

▶ pip install -q --index-url https://pypi.org/simple/ 'diffusers>=0.31.0' 'transformers==4.44.2' accelerate 'imageio[ffmpeg]' opencv-python-headless psutil supervision
▶ pip install -q --index-url https://pypi.org/simple/ 'groundingdino-py'
▶ pip install -q 'git+https://github.com/facebookresearch/sam2.git'
EMENTS FILE. If you have updated the package versions, please update the hashes. Otherwise, examine the package contents carefully; someone may have tampered with them.
          torch>=2.5.1 from https://pypi.yandex-team.ru/repo/default/download/torch/1820978/torch-2.10.0-cp312-cp312-manylinux_2_28_x86_64.whl#md5=c9914eaf754352fb9813c983cddbfced:
              Expected md5 c9914eaf754352fb9813c983cddbfced
                   Got        3bafe58c36ef743c83c0fefcf4317816
      
      
      [end of output]
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
error: subprocess-exited-with-error

× pip subprocess to install build dependencies did not 

## 2. Загрузка весов моделей

In [8]:
import os

MODELS_DIR = "/content/models"
os.makedirs(MODELS_DIR, exist_ok=True)

SAM2_CHECKPOINT  = f"{MODELS_DIR}/sam2_hiera_large.pt"
GDINO_CHECKPOINT = f"{MODELS_DIR}/groundingdino_swint_ogc.pth"
SAM2_CONFIG      = "configs/sam2.1/sam2.1_hiera_l.yaml"

def download(url, dest):
    if os.path.exists(dest):
        print(f"  ✓ уже есть: {os.path.basename(dest)} ({os.path.getsize(dest)/1e6:.0f} MB)")
        return
    print(f"  ⬇ Качаю {os.path.basename(dest)}...")
    os.system(f'wget -q --show-progress -O "{dest}" "{url}"')
    print(f"  ✓ {os.path.getsize(dest)/1e6:.0f} MB")

print("SAM2 Large:")
download(
    "https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt",
    SAM2_CHECKPOINT,
)

print("Grounding DINO:")
download(
    "https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth",
    GDINO_CHECKPOINT,
)

# Проверяем SAM2 config в пакете
import sam2 as _s2
_pkg_cfg = os.path.join(os.path.dirname(_s2.__file__), "configs", "sam2.1", "sam2.1_hiera_l.yaml")
print(f"\nSAM2 config: {'✓ найден' if os.path.exists(_pkg_cfg) else '⚠ не найден'}")
print(f"  Путь: {SAM2_CONFIG} (относительный для Hydra)")

print("\n✅ Модели готовы!")

PermissionError: [Errno 13] Permission denied: '/content'

## 3. Код пайплайна

In [ ]:
import cv2
import gc
import numpy as np
import torch
import tempfile
import os
import imageio.v2 as imageio
from PIL import Image

DEVICE = "cuda"
DTYPE  = torch.bfloat16  # CogVideoX рекомендует bfloat16

# 360×360 — оптимально для T4: CogVideoX справляется, RAM экономится.
# Если нужно качество лучше и есть GPU с >=20 GB VRAM → ставь 480×480.
COGVIDEO_H = 360
COGVIDEO_W = 360

# ── Мониторинг памяти ──────────────────────────────────────────────────
import psutil as _psutil

def mem_status(label: str = ""):
    ram_gb  = _psutil.Process().memory_info().rss / 1e9
    sys_gb  = _psutil.virtual_memory()
    vram_free, vram_total = torch.cuda.mem_get_info()
    tag = f"[{label}] " if label else ""
    print(
        f"  {tag}RAM: {ram_gb:.1f}/{sys_gb.total/1e9:.1f} GB  "
        f"| VRAM свободно: {vram_free/1e9:.1f}/{vram_total/1e9:.1f} GB"
    )

def free_mem(*objs, label: str = ""):
    """Удаляет переданные объекты, очищает кэши GPU и RAM."""
    for o in objs:
        del o
    gc.collect()
    torch.cuda.empty_cache()
    if label:
        mem_status(label)

# ── Патч совместимости для groundingdino-py + transformers >= 4.45 ─────
# groundingdino-py вызывает get_head_mask, которого нет в новых transformers.
# Патч добавляет метод обратно, если он отсутствует.
try:
    from transformers.modeling_utils import ModuleUtilsMixin
    import torch as _torch

    if not hasattr(ModuleUtilsMixin, "get_head_mask"):
        def _get_head_mask(self, head_mask, num_hidden_layers, is_attention_chunked=False):
            if head_mask is not None:
                head_mask = self._convert_head_mask_to_5d(head_mask, num_hidden_layers)
                if is_attention_chunked:
                    head_mask = head_mask.unsqueeze(-1)
            else:
                head_mask = [None] * num_hidden_layers
            return head_mask
        ModuleUtilsMixin.get_head_mask = _get_head_mask
        print("  ✓ Патч get_head_mask применён")
    else:
        print("  ✓ get_head_mask уже есть в transformers")
except Exception as _e:
    print(f"  ⚠ Патч не применён: {_e}")

print(f"\nDevice: {DEVICE} | dtype: {DTYPE}")
print(f"Разрешение для CogVideoX: {COGVIDEO_W}×{COGVIDEO_H}")
mem_status("старт")

In [ ]:
# ── Извлечение кадров ──────────────────────────────────────────────────

def extract_frames(video_path: str, target_h: int, target_w: int, max_frames=None):
    cap = cv2.VideoCapture(video_path)
    fps   = cap.get(cv2.CAP_PROP_FPS) or 24.0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame = cv2.resize(frame, (target_w, target_h))
        frames.append(frame)
        if max_frames and len(frames) >= max_frames:
            break
    cap.release()
    print(f"  Извлечено {len(frames)}/{total} кадров @ {fps:.1f} FPS")
    return frames, fps


def scan_video_for_object(video_path, gdino_model, text_prompt,
                           box_threshold, text_threshold, scan_frames=30):
    """Сканирует ВСЁ видео без загрузки всех кадров в RAM."""
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps   = cap.get(cv2.CAP_PROP_FPS) or 24.0
    cap.release()

    step = max(1, total // scan_frames)
    probe_indices = [int(i * total / scan_frames) for i in range(scan_frames)]
    print(f"  Видео: {total} кадров @ {fps:.1f} FPS | проверяю {len(probe_indices)} кадров")

    detections = []
    cap = cv2.VideoCapture(video_path)
    for fi in probe_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, fi)
        ret, frame = cap.read()
        if not ret:
            continue
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame_rgb = cv2.resize(frame_rgb, (COGVIDEO_W, COGVIDEO_H))
        result = detect_bbox(gdino_model, frame_rgb, text_prompt,
                              box_threshold, text_threshold)
        if result is not None:
            bbox, phrase, conf = result if len(result) == 3 else (*result, 0.5)
            detections.append((conf, fi, bbox, phrase, frame_rgb))
            print(f"  Кадр {fi:4d}/{total}: '{phrase}'  conf={conf:.3f} ✓")
        else:
            print(f"  Кадр {fi:4d}/{total}: не найдено")
    cap.release()
    return detections


# ── Grounding DINO ─────────────────────────────────────────────────────

def load_grounding_dino(checkpoint_path):
    import groundingdino
    from groundingdino.util.inference import load_model
    config = os.path.join(os.path.dirname(groundingdino.__file__),
                          "config", "GroundingDINO_SwinT_OGC.py")
    model = load_model(config, checkpoint_path, device=DEVICE)
    print("  ✓ Grounding DINO загружен")
    return model


def detect_bbox(model, image_np, text_prompt, box_threshold=0.35, text_threshold=0.25):
    import torchvision.transforms as T
    from groundingdino.util.inference import predict
    transform = T.Compose([
        T.Resize((800, 800)),
        T.ToTensor(),
        T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    img_t = transform(Image.fromarray(image_np))
    h, w = image_np.shape[:2]
    boxes, logits, phrases = predict(
        model=model, image=img_t, caption=text_prompt,
        box_threshold=box_threshold, text_threshold=text_threshold, device=DEVICE,
    )
    if len(boxes) == 0:
        return None
    best = int(logits.argmax())
    cx, cy, bw, bh = boxes[best].tolist()
    x1 = max(0, int((cx - bw/2) * w))
    y1 = max(0, int((cy - bh/2) * h))
    x2 = min(w, int((cx + bw/2) * w))
    y2 = min(h, int((cy + bh/2) * h))
    return (x1, y1, x2, y2), phrases[best], float(logits[best])


# ── SAM2 ───────────────────────────────────────────────────────────────

def build_sam2_image_predictor(checkpoint, config):
    from sam2.build_sam import build_sam2
    from sam2.sam2_image_predictor import SAM2ImagePredictor
    model = build_sam2(config, checkpoint, device=DEVICE)
    print("  ✓ SAM2 Image Predictor загружен")
    return SAM2ImagePredictor(model)


def get_mask_from_bbox(predictor, image_np, bbox):
    x1, y1, x2, y2 = bbox
    predictor.set_image(image_np)
    masks, _, _ = predictor.predict(
        box=np.array([x1, y1, x2, y2]), multimask_output=False,
    )
    return masks[0].astype(np.uint8) * 255


def build_sam2_video_predictor(checkpoint, config):
    from sam2.build_sam import build_sam2_video_predictor
    pred = build_sam2_video_predictor(config, checkpoint, device=DEVICE)
    print("  ✓ SAM2 Video Predictor загружен")
    return pred


def _write_frames(frames, directory):
    for i, f in enumerate(frames):
        bgr = cv2.cvtColor(f, cv2.COLOR_RGB2BGR)
        cv2.imwrite(os.path.join(directory, f"{i:05d}.jpg"), bgr,
                    [cv2.IMWRITE_JPEG_QUALITY, 95])


def track_masks_video(predictor, frames, initial_mask, seed_idx=0):
    n = len(frames)
    with tempfile.TemporaryDirectory() as tmpdir:
        _write_frames(frames, tmpdir)
        with torch.inference_mode():
            state = predictor.init_state(tmpdir)
            predictor.reset_state(state)
            predictor.add_new_mask(
                inference_state=state, frame_idx=seed_idx,
                obj_id=1, mask=initial_mask > 128,
            )
            tracked = [None] * n
            for out_idx, obj_ids, logits in predictor.propagate_in_video(state):
                if 1 not in obj_ids:
                    continue
                ch = list(obj_ids).index(1)
                binary = (logits[ch] > 0.0).squeeze(0).cpu().numpy()
                tracked[out_idx] = binary.astype(np.uint8) * 255
    empty = np.zeros(initial_mask.shape, dtype=np.uint8)
    return [m if m is not None else empty.copy() for m in tracked]


def fill_mask_gaps(masks, max_gap=2):
    n = len(masks)
    filled = [m.copy() for m in masks]
    i = 0
    while i < n:
        if filled[i].any():
            i += 1
            continue
        j = i
        while j < n and not filled[j].any():
            j += 1
        if (j - i) <= max_gap and i > 0 and j < n:
            for k in range(i, j):
                t = (k - i + 1) / (j - i + 1)
                blended = (1-t)*filled[i-1].astype(np.float32) + t*filled[j].astype(np.float32)
                filled[k] = (blended > 127).astype(np.uint8) * 255
        i = j
    return filled


print("✅ Все функции готовы!")

In [ ]:
# ── CogVideoX Pipeline ─────────────────────────────────────────────────
import gc
import psutil as _psutil

def load_cogvideo_pipeline(model_id: str, enable_cpu_offload: bool = True):
    from diffusers import CogVideoXVideoToVideoPipeline

    print(f"  Загружаю {model_id}...")
    pipe = CogVideoXVideoToVideoPipeline.from_pretrained(model_id, torch_dtype=DTYPE)

    if enable_cpu_offload:
        pipe.enable_model_cpu_offload()
        print("  ✓ CPU offload включён — веса моделей на CPU, перемещаются на GPU по мере нужды")
    else:
        pipe = pipe.to(DEVICE)

    pipe.vae.enable_tiling()    # VAE на тайлах — экономит ~2 GB VRAM
    pipe.vae.enable_slicing()   # VAE по батчам — ещё ~1 GB
    pipe.set_progress_bar_config(disable=True)
    print("  ✓ CogVideoX загружен")
    return pipe


# ── Чтение чанка из видеофайла (без загрузки всего видео в RAM) ────────

def read_chunk_from_video(video_path: str, start: int, end: int,
                           target_h: int, target_w: int) -> list:
    """Читает кадры [start, end) напрямую из файла — не нужен весь frames[] в памяти."""
    cap = cv2.VideoCapture(video_path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, start)
    chunk = []
    for _ in range(end - start):
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame = cv2.resize(frame, (target_w, target_h))
        chunk.append(frame)
    cap.release()
    return chunk


# ── Подготовка кадров (заполнение маски) ──────────────────────────────

def prefill_masked_frames(frames, masks):
    result = []
    for frame, mask in zip(frames, masks):
        if not mask.any():
            result.append(frame.copy())
            continue
        blurred = cv2.GaussianBlur(frame, (99, 99), 0)
        mask_f = (mask[..., None] / 255.0).astype(np.float32)
        filled = frame.astype(np.float32) * (1 - mask_f) + blurred.astype(np.float32) * mask_f
        result.append(filled.astype(np.uint8))
    return result


# ── Compositing: вставляем результат только в область маски ───────────

def composite_frames(original_frames, generated_frames, masks, feather_px=12):
    composed = []
    for orig, gen, mask in zip(original_frames, generated_frames, masks):
        if not mask.any():
            composed.append(orig.copy())
            continue
        k = feather_px * 2 + 1
        mask_f = cv2.GaussianBlur(mask.astype(np.float32) / 255.0, (k, k), feather_px // 2)
        mask_f = mask_f[..., None]
        result = gen.astype(np.float32) * mask_f + orig.astype(np.float32) * (1 - mask_f)
        composed.append(result.clip(0, 255).astype(np.uint8))
    return composed


# ── Обработка одного чанка ─────────────────────────────────────────────

def process_chunk(pipe, frames_chunk, masks_chunk, prompt, negative_prompt,
                  strength, num_steps, seed, target_h, target_w):
    n = len(frames_chunk)
    # CogVideoX требует нечётное кол-во кадров: 9, 17, 25, 33, 41, 49
    target_n = max(9, n if n % 8 == 1 else (n // 8) * 8 + 1)
    pad = target_n - n
    padded_frames = frames_chunk + [frames_chunk[-1]] * pad
    padded_masks  = masks_chunk  + [masks_chunk[-1]]  * pad

    prefilled = prefill_masked_frames(padded_frames, padded_masks)
    pil_input = [Image.fromarray(f) for f in prefilled]
    del prefilled  # освобождаем сразу

    generator = torch.Generator(DEVICE).manual_seed(seed)
    with torch.no_grad():
        out = pipe(
            prompt=prompt,
            negative_prompt=negative_prompt,
            video=pil_input,
            strength=strength,
            num_inference_steps=num_steps,
            guidance_scale=6.0,
            generator=generator,
            height=target_h,
            width=target_w,
            use_dynamic_cfg=True,
        )
    del pil_input

    gen_frames = []
    for f in out.frames[0]:
        arr = np.array(f) if isinstance(f, Image.Image) else (f.numpy() * 255).astype(np.uint8)
        if arr.shape[:2] != (target_h, target_w):
            arr = cv2.resize(arr, (target_w, target_h))
        gen_frames.append(arr)
    del out

    composed = composite_frames(padded_frames, gen_frames, padded_masks)
    del gen_frames
    return composed[:n]  # только реальные (без паддинга)


# ── Потоковая обработка: читает из файла, пишет в файл чанк за чанком ─

def process_video_streaming(pipe, video_path, masks, prompt, negative_prompt,
                             strength, num_steps, seed, output_path, fps,
                             chunk_size=49, overlap=8,
                             target_h=360, target_w=360):
    """
    Потоковая обработка: кадры читаются из файла чанк за чанком (не из RAM),
    результат пишется сразу на диск. Пик RAM = один чанк × 2 (orig + result).
    """
    n = len(masks)
    if chunk_size % 2 == 0:
        chunk_size += 1
    step = chunk_size - overlap
    starts = list(range(0, n, step))

    # Буфер только для overlap-зоны между чанками (не всё видео)
    overlap_buf = {}  # global_idx -> (accumulated_frame_f32, total_weight)

    writer = imageio.get_writer(output_path, fps=fps, macro_block_size=1)
    next_write_idx = 0  # следующий кадр для записи

    def flush_ready_frames(up_to: int):
        """Записываем все кадры вплоть до up_to (не включая) которые уже готовы."""
        nonlocal next_write_idx
        while next_write_idx < up_to:
            if next_write_idx in overlap_buf:
                f_f32, w = overlap_buf.pop(next_write_idx)
                writer.append_data((f_f32 / w).clip(0, 255).astype(np.uint8))
            next_write_idx += 1

    for ci, start in enumerate(starts):
        end = min(start + chunk_size, n)
        chunk_masks = masks[start:end]

        # Зона без overlap-перекрытия от следующего чанка — можно сбросить на диск
        safe_end = start if ci == 0 else min(start + overlap // 2, end)
        flush_ready_frames(safe_end)

        if not any(m.any() for m in chunk_masks):
            # Объект не виден: читаем оригинал и пишем без изменений
            orig_chunk = read_chunk_from_video(video_path, start, end, target_h, target_w)
            for local_i, fi in enumerate(range(start, end)):
                if fi < n:
                    frame = orig_chunk[local_i]
                    if fi in overlap_buf:
                        acc, w = overlap_buf[fi]
                        overlap_buf[fi] = (acc + frame.astype(np.float32), w + 1.0)
                    else:
                        overlap_buf[fi] = (frame.astype(np.float32), 1.0)
            del orig_chunk
            print(f"  Чанк {ci+1}/{len(starts)}: кадры {start}-{end-1} — объект не виден, пропускаю")
            continue

        print(f"  Чанк {ci+1}/{len(starts)}: кадры {start}-{end-1} → CogVideoX...")
        orig_chunk = read_chunk_from_video(video_path, start, end, target_h, target_w)
        chunk_result = process_chunk(
            pipe, orig_chunk, chunk_masks,
            prompt, negative_prompt, strength, num_steps, seed + ci,
            target_h, target_w,
        )
        del orig_chunk

        # Записываем в overlap-буфер с взвешенным усреднением на стыках
        chunk_len = end - start
        for local_i, fi in enumerate(range(start, end)):
            if fi >= n:
                break
            edge_dist = min(local_i, chunk_len - 1 - local_i)
            w = min(1.0, (edge_dist + 1) / max(overlap, 1))
            frame_f32 = chunk_result[local_i].astype(np.float32)
            if fi in overlap_buf:
                acc, prev_w = overlap_buf[fi]
                overlap_buf[fi] = (acc * prev_w + frame_f32 * w, prev_w + w)
            else:
                overlap_buf[fi] = (frame_f32 * w, w)
        del chunk_result

        gc.collect()
        torch.cuda.empty_cache()
        buf_mb = sum(v[0].nbytes for v in overlap_buf.values()) // (1024 * 1024)
        ram_gb = _psutil.Process().memory_info().rss / 1e9
        vram_free = torch.cuda.mem_get_info()[0] / 1e9
        print(f"    RAM: {ram_gb:.1f} GB  |  буфер: ~{buf_mb} MB  |  VRAM свободно: {vram_free:.1f} GB")

    # Сбрасываем оставшиеся кадры
    flush_ready_frames(n)
    writer.close()
    print(f"  ✓ Видео записано: {output_path}")


print("✅ CogVideoX функции готовы!")

## 4. Загрузка видео

In [ ]:
# ── Вариант A: загрузить файл с компьютера ────────────────────────────
from google.colab import files
uploaded = files.upload()
INPUT_VIDEO = list(uploaded.keys())[0]
print(f"Загружено: {INPUT_VIDEO}  ({os.path.getsize(INPUT_VIDEO)/1e6:.1f} MB)")

In [ ]:
# ── Вариант B: Google Drive ───────────────────────────────────────────
# from google.colab import drive
# drive.mount('/content/drive')
# INPUT_VIDEO = "/content/drive/MyDrive/my_video.mp4"

## 5. Параметры

In [ ]:
# ┌─────────────────────────────────────────────────────────────────────┐
# │                     НАСТРОЙКИ                                       │
# └─────────────────────────────────────────────────────────────────────┘

# Что найти (Grounding DINO)
DETECT_PROMPT  = "clock watch"
BOX_THRESHOLD  = 0.35
TEXT_THRESHOLD = 0.25
SCAN_FRAMES    = 30  # сканируем всё видео целиком

# Что сгенерировать (CogVideoX prompt)
# Совет: описывай объект В ДВИЖЕНИИ — CogVideoX понимает динамику
PROMPT = (
    "a ceramic mug with space galaxy design, sitting on a surface, "
    "photorealistic, cinematic lighting, 4k"
)
NEGATIVE_PROMPT = "distorted, blurry, extra objects, low quality, watermark"

# Модель CogVideoX
# "THUDM/CogVideoX-2b" — меньше (6 GB), быстрее, подходит для T4
# "THUDM/CogVideoX-5b" — качественнее, нужен A100 (40 GB VRAM)
COGVIDEO_MODEL = "THUDM/CogVideoX-2b"

# Параметры генерации
STRENGTH       = 0.8   # насколько сильно менять кадры (0.5=мало, 0.95=сильно)
NUM_STEPS      = 50    # шаги диффузии (25=быстро, 50=качественно)
SEED           = 42

# Параметры чанков
# CogVideoX обрабатывает строго 2k+1 кадров: 9, 17, 25, 33, 41, 49
CHUNK_SIZE     = 25    # 25 кадров — меньше нагрузка на RAM/VRAM чем 49
CHUNK_OVERLAP  = 4     # кадров перекрытия между чанками (для плавного сшивания)

# CPU offload: True по умолчанию — безопасно для T4 (15 GB VRAM).
# Ставь False только если VRAM достаточно (>= 20 GB) и хочешь ускорить.
ENABLE_CPU_OFFLOAD = True

# DEBUG: None = обработать всё видео. Число = только первые N кадров.
DEBUG_MAX_FRAMES = None

OUTPUT_VIDEO = "/content/output_cogvideo.mp4"

print("Параметры:")
print(f"  Что найти:   '{DETECT_PROMPT}'")
print(f"  Промпт:      '{PROMPT[:60]}...'")
print(f"  Модель:      {COGVIDEO_MODEL}")
print(f"  Strength:    {STRENGTH}  |  Steps: {NUM_STEPS}")
print(f"  Chunk:       {CHUNK_SIZE} кадров, overlap={CHUNK_OVERLAP}")
print(f"  Обработка:   {'ВСЁ видео' if not DEBUG_MAX_FRAMES else f'{DEBUG_MAX_FRAMES} кадров (DEBUG)'}")

## 6. Запуск пайплайна

In [ ]:
# ── Шаг 1: Поиск объекта по всему видео ───────────────────────────────
print("=" * 60)
print("Шаг 1: Поиск объекта (Grounding DINO — сканирую всё видео)")
print("=" * 60)
mem_status("до DINO")

gdino = load_grounding_dino(GDINO_CHECKPOINT)
detections = scan_video_for_object(
    INPUT_VIDEO, gdino, DETECT_PROMPT, BOX_THRESHOLD, TEXT_THRESHOLD, SCAN_FRAMES
)

if not detections:
    # Удаляем DINO перед исключением чтобы не занимать RAM
    del gdino; gc.collect(); torch.cuda.empty_cache()
    raise RuntimeError(
        f"Объект '{DETECT_PROMPT}' не найден.\n"
        f"Попробуй: снизь BOX_THRESHOLD до 0.20, или измени DETECT_PROMPT"
    )

detections.sort(reverse=True)
best_conf, seed_frame_idx, found_bbox, found_phrase, seed_frame_rgb = detections[0]
print(f"\n  ★ Лучший: '{found_phrase}' на кадре {seed_frame_idx}  conf={best_conf:.3f}")
if best_conf < 0.40:
    print(f"  ⚠ Уверенность невысокая — подними BOX_THRESHOLD если нашёлся не тот объект")

# Grounding DINO больше не нужен — удаляем ДО загрузки кадров и SAM2
del gdino
gc.collect()
torch.cuda.empty_cache()
mem_status("после удаления DINO")

In [ ]:
# ── Шаг 2: Извлечение всех кадров ─────────────────────────────────────
print("=" * 60)
print("Шаг 2: Извлечение кадров")
print("=" * 60)

frames, fps = extract_frames(
    INPUT_VIDEO, COGVIDEO_H, COGVIDEO_W, max_frames=DEBUG_MAX_FRAMES
)
local_seed_idx = min(seed_frame_idx, len(frames) - 1)
print(f"  Seed-кадр: {local_seed_idx}")

# ── Шаг 3: Начальная маска (SAM2 Image) ───────────────────────────────
print("\n" + "=" * 60)
print("Шаг 3: Создание начальной маски (SAM2)")
print("=" * 60)

sam_img = build_sam2_image_predictor(SAM2_CHECKPOINT, SAM2_CONFIG)
initial_mask = get_mask_from_bbox(sam_img, frames[local_seed_idx], found_bbox)
del sam_img
torch.cuda.empty_cache()
print(f"  ✓ Пикселей в маске: {initial_mask.sum() // 255}")

# Превью
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(frames[local_seed_idx])
x1, y1, x2, y2 = found_bbox
axes[0].add_patch(plt.Rectangle((x1,y1), x2-x1, y2-y1,
                                  fill=False, edgecolor='lime', linewidth=3))
axes[0].set_title(f"'{found_phrase}'  conf={best_conf:.3f}", fontsize=13)
axes[0].axis('off')
overlay = frames[local_seed_idx].copy()
overlay[initial_mask > 128] = [255, 80, 80]
axes[1].imshow((0.55*frames[local_seed_idx] + 0.45*overlay).astype(np.uint8))
axes[1].set_title("Маска SAM2 (красное = будет заменено)", fontsize=13)
axes[1].axis('off')
plt.tight_layout()
plt.show()
print("\n  ↑ Проверь: красная зона должна покрывать нужный объект.")

In [ ]:
# ── Шаг 4: Отслеживание маски (SAM2 Video) ────────────────────────────
print("=" * 60)
print("Шаг 4: Отслеживание объекта (SAM2 Video Predictor)")
print("=" * 60)
mem_status("до SAM2 Video")

sam_vid = build_sam2_video_predictor(SAM2_CHECKPOINT, SAM2_CONFIG)
tracked_masks = track_masks_video(sam_vid, frames, initial_mask, local_seed_idx)
del sam_vid
gc.collect()
torch.cuda.empty_cache()

tracked_masks = fill_mask_gaps(tracked_masks, max_gap=2)
visible = sum(m.any() for m in tracked_masks)
print(f"  ✓ Объект виден в {visible}/{len(frames)} кадрах")

# Освобождаем frames из RAM — больше не нужны здесь.
# CogVideoX будет читать кадры напрямую из файла чанк за чанком.
n_frames_total = len(frames)
del frames, initial_mask
gc.collect()
torch.cuda.empty_cache()
mem_status("после удаления frames")
print("  ✓ frames удалены из RAM — CogVideoX будет читать из файла")

# ── Шаг 5: Загрузка CogVideoX ─────────────────────────────────────────
print("\n" + "=" * 60)
print("Шаг 5: Загрузка CogVideoX")
print("=" * 60)

cogvideo_pipe = load_cogvideo_pipeline(COGVIDEO_MODEL, ENABLE_CPU_OFFLOAD)
mem_status("после загрузки CogVideoX")

In [ ]:
# ── Шаг 6: Потоковая обработка (CogVideoX читает из файла, пишет на диск)
import time, gc

print("=" * 60)
print("Шаг 6: Замена объекта (CogVideoX — потоковая обработка)")
print("=" * 60)

step = CHUNK_SIZE - CHUNK_OVERLAP
n_chunks = max(1, (n_frames_total + step - 1) // step)
est_min = n_chunks * NUM_STEPS * 0.6 / 60
print(f"  Всего кадров:  {n_frames_total}")
print(f"  Чанков:        {n_chunks}  ({CHUNK_SIZE} кадров, overlap={CHUNK_OVERLAP})")
print(f"  Оценка:        ~{est_min:.0f}–{est_min*1.5:.0f} мин на T4")
print(f"  RAM-стратегия: кадры читаются из файла, результат пишется сразу на диск")
print()

t0 = time.time()

# Потоковая обработка: frames НЕ в памяти, всё читается из INPUT_VIDEO
process_video_streaming(
    cogvideo_pipe, INPUT_VIDEO, tracked_masks,
    PROMPT, NEGATIVE_PROMPT,
    strength=STRENGTH,
    num_steps=NUM_STEPS,
    seed=SEED,
    output_path=OUTPUT_VIDEO,
    fps=fps,
    chunk_size=CHUNK_SIZE,
    overlap=CHUNK_OVERLAP,
    target_h=COGVIDEO_H,
    target_w=COGVIDEO_W,
)

elapsed = time.time() - t0
print(f"\n✅ Обработка завершена за {elapsed/60:.1f} мин")

In [ ]:
# ── Шаг 7: Превью результата ──────────────────────────────────────────
# Видео уже записано на диск в процессе обработки.
# Кадры читаются поштучно — не загружаем всё видео в RAM.
import matplotlib.pyplot as plt

size_mb = os.path.getsize(OUTPUT_VIDEO) / 1e6
print(f"✅ Видео готово: {OUTPUT_VIDEO}  ({size_mb:.1f} MB)")

preview_global_idxs = [int(n_frames_total * t) for t in [0.1, 0.35, 0.6, 0.85]]

cap_orig   = cv2.VideoCapture(INPUT_VIDEO)
cap_result = cv2.VideoCapture(OUTPUT_VIDEO)

fig, axes = plt.subplots(len(preview_global_idxs), 2,
                          figsize=(12, 5 * len(preview_global_idxs)))
for row, fi in enumerate(preview_global_idxs):
    fi = min(fi, n_frames_total - 1)

    cap_orig.set(cv2.CAP_PROP_POS_FRAMES, fi)
    ret, orig_frame = cap_orig.read()
    if ret:
        orig_frame = cv2.cvtColor(orig_frame, cv2.COLOR_BGR2RGB)
        orig_frame = cv2.resize(orig_frame, (COGVIDEO_W, COGVIDEO_H))
    else:
        orig_frame = np.zeros((COGVIDEO_H, COGVIDEO_W, 3), dtype=np.uint8)

    cap_result.set(cv2.CAP_PROP_POS_FRAMES, fi)
    ret, res_frame = cap_result.read()
    if ret:
        res_frame = cv2.cvtColor(res_frame, cv2.COLOR_BGR2RGB)
        res_frame = cv2.resize(res_frame, (COGVIDEO_W, COGVIDEO_H))
    else:
        res_frame = np.zeros((COGVIDEO_H, COGVIDEO_W, 3), dtype=np.uint8)

    axes[row][0].imshow(orig_frame)
    axes[row][0].set_title(f"Кадр {fi} — ДО", fontsize=13)
    axes[row][0].axis('off')
    axes[row][1].imshow(res_frame)
    axes[row][1].set_title(f"Кадр {fi} — ПОСЛЕ (CogVideoX)", fontsize=13)
    axes[row][1].axis('off')

cap_orig.release()
cap_result.release()

plt.suptitle("До / После", fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Шаг 8: Скачать результат ──────────────────────────────────────────
from google.colab import files
files.download(OUTPUT_VIDEO)
print("✅ Файл отправлен в загрузки!")

---

## Советы по настройке

### Параметр `STRENGTH`
- `0.5–0.6` — лёгкое изменение, объект будет похож на оригинал
- `0.7–0.8` — стандарт, хороший баланс
- `0.85–0.95` — агрессивное изменение, максимальное следование промпту

### Параметр `CHUNK_SIZE`
CogVideoX принимает строго `2k+1` кадров: **9, 17, 25, 33, 41, 49**
- `49` — максимальный (~6 сек), лучшая временная связность
- `25` — меньше памяти, быстрее

### Промпт для CogVideoX
В отличие от SDXL, CogVideoX понимает **динамику**. Описывай поведение:
- ❌ `"ceramic mug"` — слишком коротко
- ✅ `"a ceramic mug with galaxy design resting on a wooden table, soft studio lighting, photorealistic, 4k"`

### Если мало VRAM (OOM)
1. Поставь `ENABLE_CPU_OFFLOAD = True`
2. Уменьши `CHUNK_SIZE` до `25`
3. Поставь `DEBUG_MAX_FRAMES = 49` для быстрого теста

### Объект не найден
- Снизь `BOX_THRESHOLD` до `0.20`
- Попробуй простые слова: `mug`, `cup`, `watch`, `bottle`, `clock`